## ECC implementation and visualization

Gerard Consuelo, Atay Moya, Larry Shields

### Overview

---

RSA is the most widely accepted public-key scheme. It is used for encryption, digital signatures, and symmetric key exchange. Though quite powerful in that it is computationally impossible to reverse its process, it is very computational heavy. Its key sizes are enormous at around 512 bits, and must keep increasing with the rapid development of technology. 

Elliptical Curve Cryptography (ECC) is a method of cryptography based on elliptical curves and the difficulty in computing the elliptic curve discrete logarithmic problem. It offers the same security as RSA, with the differnece that it is extremely efficient for smaller bit sizes, where a 256 -bit key for ECC provides as much security as a 3072-bit key using RSA.


### The Elliptic Curve Discrete Logarithmic Problem

---

In RSA, recall that the core equation we have was starting with two large prime numbers, p and q, to get our modulo. 
$$N = p \cdot q$$

Then we would use Euler's Totient function (p-1)(q-1) to narrow it down to a finite amount of numbers. 
$$\phi(N) = (p-1)(q-1)$$

Then, we pick a public key e in between 1 and the totient. 
$$1 < e < \phi(N) \quad \text{where} \quad \gcd(e, \phi(N)) = 1$$

To achieve the private key d, we would find the public key's inverse through the modulo stuff. 
$$d \equiv e^{-1} \pmod{\phi(N)}$$

To encrypt, the ciphertext (c) is m^e and to decrypt, the plaintext (m) should be the result from c^d.
$$\text{Encryption: } c \equiv m^e \pmod N$$
$$\text{Decryption: } m \equiv c^d \pmod N$$

ECC follows a similar process, but with points on the elliptic curve.
$$y^2 \equiv x^3 + ax + b \pmod p$$

The core equation we start with for ECC is k * P = Q, which is the elliptic curve discrete logarithmic problem. 
$$Q = k \cdot P$$
*(The ECDLP states: Given points $P$ and $Q$, it is computationally infeasible to find the scalar $k$.)*

In this equation, k is a scalar, and P and Q are points on a given curve. 
$$k \in \{1, 2, \dots, n-1\} \quad \text{(where } n \text{ is the order of the curve)}$$
$$P, Q \in E(\mathbb{F}_p)$$

For encryption, k is our private key, Q is our public key, and P is a given constant starting point.
$$\text{Private Key: } k$$
$$\text{Starting Point (Generator): } P = (x_p, y_p)$$
$$\text{Public Key: } Q = (x_q, y_q)$$


### Algebra

---

Before we go over algebra, we will first go over the parameters. 
The domain parameters for an elliptic curve over $F_p$ are $p$, $a$, $b$, $G$, $n$, and $h$.

$p$ is the prime number that defines the finite field $F_p$.
$a$ and $b$ are the parameters that define the curve $y^2 = x^3 + ax + b$ over $F_p$.
$G$ is the generator point $(x_G, y_G)$, a point on the elliptic curve chosen for cryptographic operations.
$n$ is the order of the elliptic curve.
The scalar for point multiplication is chosen as a number between $0$ and $n - 1$.
$h$ is the cofactor, where $h = \#E(F_p) / n$. $\#E(F_p)$ is the number of points on an elliptic curve.

With those parameters defined, we can look at the algebra of an elliptic curve.
The key operations we need to examine are point addition and point doubling.

**Point Addition**
This operation is defined as $R = P + Q$, where $P$, $Q$, and the resulting point $R$ all lie on the curve, or are the identity element, denoted as $\mathcal{O}$ (often referred to as the Point at Infinity). 

Elliptic curves have a geometric property where drawing a straight line through any 2 points ($P$ and $Q$) will intersect the curve at exactly one third point, which we call $-R$. To find the actual result of $P + Q$, you take that third intersection point and **reflect it across the x-axis** to get $R$. 

An edge case occurs when $Q = -P$ (meaning they share the same x-coordinate but have opposite y-coordinates). The line through them is purely vertical, meaning it never intersects the curve a third time. Instead, we say it intersects at infinity, resulting in our identity value: $P + (-P) = \mathcal{O}$. 

Assuming our curve follows the standard form $y^2 = x^3 + ax + b$, and $P = (x_1, y_1)$ and $Q = (x_2, y_2)$, here are the equations to find $R = (x_3, y_3)$:

$$\lambda = \frac{y_2 - y_1}{x_2 - x_1}$$
$$x_3 = \lambda^2 - x_1 - x_2$$
$$y_3 = \lambda(x_1 - x_3) - y_1$$

**Point Doubling**
Building on top of addition, this is simply calculating $2P = P + P$. Geometrically, since $P$ and $Q$ are the exact same point, you cannot draw a standard line between them. Instead, you draw the **tangent line** to the curve exactly at point $P$. This tangent line will intersect the curve at another point, $-2P$. Just like in addition, you reflect this point across the x-axis to find $2P$.

$2P$ results in the identity value $\mathcal{O}$ if $P$ is already the identity value, or if the y-coordinate of $P$ is exactly $0$ (which makes the tangent line perfectly vertical).

Using the same curve parameters, here are the equations for point doubling to find $2P = (x_3, y_3)$:

$$\lambda = \frac{3x_1^2 + a}{2y_1}$$
$$x_3 = \lambda^2 - 2x_1$$
$$y_3 = \lambda(x_1 - x_3) - y_1$$

With these algebraic pieces, we now know the mechanics behind scalar multiplication, like $k \cdot P$ on an elliptic curve. This means adding $P$ to itself $k$ times:
$$P + P + P + \dots \text{ (k times)}$$

### Finite Field

---

However, the numbers can get way too large, leading to round-off errors and sluggishness. To remedy this, we use a modulo over $p$, allowing an infinite number of inputs but a finite number of outputs. There will be overlaps, making it much harder to reverse.

So the equations will look like this:

**Point Addition ($R = P + Q$)**

points $P = (x_1, y_1)$ and $Q = (x_2, y_2)$:

$$\lambda = (y_2 - y_1) \cdot (x_2 - x_1)^{-1} \pmod{p}$$
$$x_3 = \lambda^2 - x_1 - x_2 \pmod{p}$$
$$y_3 = \lambda(x_1 - x_3) - y_1 \pmod{p}$$

**Point Doubling ($2P$)**

point $P = (x_1, y_1)$ (where $y_1 = 0$):

$$\lambda = (3x_1^2 + a) \cdot (2y_1)^{-1} \pmod{p}$$
$$x_3 = \lambda^2 - 2x_1 \pmod{p}$$
$$y_3 = \lambda(x_1 - x_3) - y_1 \pmod{p}$$

### Modular Arithmetic

---

Because now we are in a finite field, the algebra changes as well. Here are the ones we are using above and the explanation. Notice there is no division, only multiplication. For these operations, we assume that the inputs are all within the modulo already.

**Addition**
When adding two numbers in a finite field, the worst-case bit length increases by 1 (e.g., an 8-bit number might overflow into a 9-bit number). To keep the result within the field, we simply subtract the modulo $p$ from the addition.
* **Example:** $15 + 20 = 35$. Since $35 > 23$, we do $35 - 23 = 12$.

**Subtraction**
Similar to addition, subtraction worst case brings the bit length down or the value to become negative. To keep the result within the field, we simply add the modulo $p$ to the result.
* **Example:** $15 - 20 = -5$. Since $-5 < 0$, we do $-5 + 23 = 18$.


**Multiplication (Double-and-Add Algorithm)**

Instead of multiplying two large numbers directly which can cause the bit length to overflow, we can rely on bitwise properties using the "double-and-add" method. We follow these steps.
1.  Get the number we multiply by, m, and get its bit representation.
2.  Loop through its bits
3.  if the bit is "on", then we add m to a separate value.
4.  double m (since every shift is *2 basically)

$15 \cdot 20 \bmod 23$

$B = 20$ in binary is **`10100`**

`Total = 0` (This will accumulate our final answer)
`Current_A = 15` (We will continuously double this value modulo 23)

**Bit 1 (Rightmost bit) = 0**
  * Since the bit is **0**, we DO NOT add `Current_A` to our `Total`.
  * `Total` remains $0$.
  * DOUBLE `Current_A` for the next step: $(15 \times 2) \pmod{23} = 30 \pmod{23} = 7$.

**Bit 2 = 0**
  * Since the bit is **0**, we DO NOT add `Current_A` to our `Total`.
  * `Total` remains $0$.
  * DOUBLE `Current_A` for the next step: $(7 \times 2) \pmod{23} = 14 \pmod{23} = 14$.

**Bit 3 = 1**
  * Since the bit is **1**, we ADD `Current_A` to our `Total`.
  * `Total` = $(0 + 14) \pmod{23} = 14$.
  * DOUBLE `Current_A` for the next step: $(14 \times 2) \pmod{23} = 28 \pmod{23} = 5$.

**Bit 4 = 0**
  * Since the bit is **0**, we DO NOT add `Current_A` to our `Total`. 
  * `Total` remains $14$.
  * DOUBLE `Current_A` for the next step: $(5 \times 2) \pmod{23} = 10 \pmod{23} = 10$.

**Bit 5 (Leftmost bit) = 1**
  * Since the bit is **1**, we ADD `Current_A` to our `Total`.
  * `Total` = $(14 + 10) \pmod{23} = 24 \pmod{23} = 1$.
  * *(We stop doubling `Current_A` here as we have processed all the bits).*



**Division using The Extended Euclidean Algorithm**

In modular arithmetic (a finite field defined by a prime $P$), standard division does not exist. Instead of dividing $A$ by $B$ (calculating $A / B$), we must compute $A \cdot B^{-1} \pmod P$, where $B^{-1}$ is the **modular multiplicative inverse** of $B$. 

Finding this inverse relies on the **Extended Euclidean Algorithm (EEA)**. Here is how the math unfolds:

* At its core, the standard Euclidean algorithm efficiently computes the Greatest Common Divisor (GCD) of two numbers. It is based on the principle that the GCD of two numbers does not change if the larger number is replaced by its remainder when divided by the smaller number. 
  * To find the GCD of two numbers, $A$ and $B$ (assuming $A > B$):
    1. Divide $A$ by $B$ to find the quotient $Q$ and the remainder $R$. (Mathematically: $A = Q \cdot B + R$).
    2. If the remainder $R = 0$, then your current $B$ is the GCD. You are done.
    3. If $R \neq 0$, shift your numbers: replace $A$ with $B$, and replace $B$ with $R$. 
    4. Repeat the division with the new $A$ and $B$ until the remainder hits $0$. The last non-zero remainder (which is the divisor at the final step) is the GCD.
  * **Example:** $\gcd(252, 105)$.
    * **Step 1:** $252 \div 105 = 2$ with a remainder of **$42$**. 
      *(Equation: $252 = 2 \cdot 105 + 42$)*
    * **Step 2:** Shift the numbers. Our new division is $105 \div 42$.
      $105 \div 42 = 2$ with a remainder of **$21$**. 
      *(Equation: $105 = 2 \cdot 42 + 21$)*
    * **Step 3:** Shift the numbers again. Our new division is $42 \div 21$.
      $42 \div 21 = 2$ with a remainder of **$0$**. 
      *(Equation: $42 = 2 \cdot 21 + 0$)*
    * **Result:** Since the remainder is now $0$, the algorithm stops. The last non-zero remainder (our final divisor) is $21$. Therefore, $\gcd(252, 105) = 21$.
* The *Extended* version of the algorithm goes a step further by calculating the coefficients (often called Bézout coefficients) that satisfy Bézout's Identity. This theorem states that for any two integers $B$ and $P$, there exist integer coefficients $m$ and $n$ such that:
  $$B \cdot m + P \cdot n = \gcd(B, P)$$
* In elliptic curve cryptography, our modulo $P$ is a prime number. Because $P$ is prime, it has no positive divisors other than 1 and itself. Therefore, as long as $B$ is not $0$, the GCD of $B$ and $P$ is guaranteed to be exactly 1. Bézout's identity simplifies perfectly to:
  $$B \cdot m + P \cdot n = 1$$
* If we take the modulo $P$ of both sides of this equation, the term $P \cdot n$ becomes exactly $0$ (because any multiple of $P$ is $0 \pmod P$). The equation isolates our terms to:
  $$B \cdot m \equiv 1 \pmod P$$
* By mathematical definition, a number multiplied by its inverse equals 1. Therefore, the coefficient $m$ (which the EEA calculates for us) is exactly our modular inverse, $B^{-1}$! 

Once we use the EEA to find $m$, our original division problem ($A / B$) simply becomes an easy multiplication problem: $A \times m \pmod P$.